# 00 · Pembekuan protokol dan pemeriksaan input

**Pertanyaan penelitian.** Seberapa besar perubahan kinerja klasifikasi arah XAUUSD H1 (BUY, HOLD, SELL) ketika
fitur teknikal (K2 − K1) dan fitur makroekonomi (K3 − K2) ditambahkan, pada Regresi Logistik Multinomial dan XGBoost?

Rangkaian notebook ini adalah **pelaksanaan Fase 4** sesuai Bab 3 proposal final P0
(`docs/latex-proposal/chapters/bab3-metodologi.tex`). Setiap notebook menyebut subbab Bab 3 yang dipenuhinya.

| Notebook | Isi | Bab 3 |
|---|---|---|
| 00 | Konfigurasi beku, checksum input, catatan lingkungan | 3.3, 3.4, 3.12 |
| 01 | Audit harga dan klasifikasi gap | 3.6.1 |
| 02 | Waktu keputusan, volatilitas σ24, label BUY/HOLD/SELL | 3.5 |
| 03 | Fitur harga (K1) dan teknikal RSI/MACD (K2) | 3.7.1, 3.7.2 |
| 04 | Audit kalender dan fitur makro (K3) | 3.6.2, 3.7.3 |
| 05 | Skema walk-forward, blok validasi, purge | 3.9.1, 3.9.2 |
| 06 | Praproses, model, penyetelan, prediksi | 3.9.2–3.9.4 |
| 07 | Metrik, perubahan berpasangan, sensitivitas | 3.10 |
| 08 | Verifikasi implementasi dan determinisme | 3.12 |

**Aturan main**
1. Konfigurasi `configs/phase4-p0.toml` dan kode notebook di-commit **sebelum** skor periode uji dibaca.
2. Data vendor dan hasil per bar disimpan di `data/interim/phase4-p0/` yang diabaikan Git.
3. Tidak ada keputusan (fitur, ambang, parameter) yang diambil dari skor periode uji.
4. Lingkup hanya MetaQuotes-Demo; sumber Ultima dan policy holiday-reset berada di luar protokol P0.

Jalankan notebook berurutan 00 → 08. Setiap notebook membaca keluaran notebook sebelumnya.

In [1]:
import tomllib
from pathlib import Path

# Cari akar repositori (folder yang memuat pyproject.toml) agar notebook bisa dijalankan dari mana saja.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
CFG = tomllib.loads((ROOT / "configs/phase4-p0.toml").read_text(encoding="utf-8"))
OUT = ROOT / "data/interim/phase4-p0"     # folder hasil; diabaikan Git karena memuat data vendor
OUT.mkdir(parents=True, exist_ok=True)

## 1. Isi konfigurasi beku

Semua angka protokol dibaca dari satu berkas. Bila ada yang perlu diubah, ubah berkas itu, commit, lalu jalankan ulang
dari notebook 00 — jangan mengubah angka di dalam notebook.

In [2]:
import hashlib
import json

import pandas as pd


def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


cfg_path = ROOT / "configs/phase4-p0.toml"
print("Konfigurasi :", cfg_path.relative_to(ROOT))
print("SHA-256     :", sha256(cfg_path))
print("Ambang utama:", CFG["label"]["threshold_utama"], "· sensitivitas:", CFG["label"]["threshold_sensitivitas"])
print("Pemanasan   : observasi dipakai mulai bar ke-", CFG["fitur"]["first_usable_segment_bar"], "dalam segmen", sep="")

pd.DataFrame(CFG["validasi"]["periode_uji"], columns=["periode", "awal (termasuk)", "akhir (tidak termasuk)"])

Konfigurasi : configs/phase4-p0.toml
SHA-256     : c8075480fcacd510d2f3e775d8d224d0a6b75af2af5876e142513adba92ad082
Ambang utama: 0.5 · sensitivitas: [0.25, 1.0]
Pemanasan   : observasi dipakai mulai bar ke-51dalam segmen


,periode,awal (termasuk),akhir (tidak termasuk)
0,2023-H1,2023-01-01T00:00:00Z,2023-07-01T00:00:00Z
1,2023-H2,2023-07-01T00:00:00Z,2024-01-01T00:00:00Z
2,2024-H1,2024-01-01T00:00:00Z,2024-07-01T00:00:00Z
3,2024-H2,2024-07-01T00:00:00Z,2025-01-01T00:00:00Z
4,2025-H1,2025-01-01T00:00:00Z,2025-07-01T00:00:00Z
5,2025-H2,2025-07-01T00:00:00Z,2026-01-01T00:00:00Z
6,2026-Jan-Jul,2026-01-01T00:00:00Z,2026-08-01T00:00:00Z


## 2. Checksum input

Input harus sama persis (byte demi byte) dengan arsip yang dibekukan. Bila checksum berbeda, notebook berhenti.

In [3]:
cek = []
for nama, item in CFG["inputs"].items():
    path = ROOT / item["path"]
    aktual = sha256(path)
    cek.append({"input": nama, "berkas": item["path"], "cocok": aktual == item["sha256"], "sha256": aktual[:16] + "…"})
cek = pd.DataFrame(cek)
assert cek["cocok"].all(), "Ada input yang tidak sama dengan arsip beku"
cek

,input,berkas,cocok,sha256
0,harga,data/research/windows-feasibility-2026-08-09/m...,True,30684142acadd612…
1,kalender,data/raw/macro_calendar/mt5-economic-calendar-...,True,0e538e47593a57da…
2,jadwal_gap,configs/phase4-market-gaps.toml,True,e77fd095781bbbcc…


## 3. Catatan lingkungan dan Git

Bab 3 Subbab 3.12 meminta versi perangkat lunak, benih acak, dan identitas kode dicatat bersama artefak.

In [4]:
import platform
import subprocess
import sys

import matplotlib
import numpy
import pyarrow
import sklearn
import xgboost


def git(*args):
    """Jalankan perintah git; di luar repositori git (misalnya salinan ZIP) hasilnya "tidak tersedia"."""
    hasil = subprocess.run(["git", *args], cwd=ROOT, capture_output=True, text=True, check=False)
    return hasil.stdout.strip() if hasil.returncode == 0 else "tidak tersedia"


lingkungan = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "numpy": numpy.__version__,
    "pandas": pd.__version__,
    "pyarrow": pyarrow.__version__,
    "scikit_learn": sklearn.__version__,
    "xgboost": xgboost.__version__,
    "matplotlib": matplotlib.__version__,
}
kode = {
    "branch": git("rev-parse", "--abbrev-ref", "HEAD"),
    "commit": git("rev-parse", "HEAD"),
    "worktree_bersih": git("status", "--porcelain", "--", "configs", "notebooks") == "",  # False juga bila git tidak tersedia
}
manifest = {
    "dibuat_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "konfigurasi": {"path": "configs/phase4-p0.toml", "sha256": sha256(cfg_path)},
    "input": {r.input: {"path": r.berkas, "sha256": sha256(ROOT / r.berkas)} for r in cek.itertuples()},
    "seed": CFG["experiment"]["seed"],
    "lingkungan": lingkungan,
    "kode": kode,
    "skor_dibaca": False,
}
(OUT / "00_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(json.dumps({**lingkungan, **kode}, indent=2))

{
  "python": "3.14.5",
  "platform": "macOS-27.0.1-arm64-arm-64bit-Mach-O",
  "numpy": "2.5.3",
  "pandas": "3.0.5",
  "pyarrow": "23.0.1",
  "scikit_learn": "1.9.0",
  "xgboost": "2.1.4",
  "matplotlib": "3.11.2",
  "branch": "research/phase-4-experiments",
  "commit": "0a671a3ce6586ce0362ee337e0cc93eaca980cb0",
  "worktree_bersih": true
}


`worktree_bersih = True` berarti konfigurasi dan notebook yang dijalankan sama dengan yang ada di commit. Untuk run resmi,
nilai ini harus `True`.